# SharePoint mango label audit

This exploratory notebook samples 20 HEIC images from the SharePoint fruit-image folder. It renders the images, optionally runs a locally installed OCR backend, and creates a review table for recording whether variety and specimen labels are present and readable.

The notebook never edits the raw files. OCR output is only a suggestion; the human review columns are the fields that should eventually feed an annotation manifest.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import re
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps

# Locate the repository whether Jupyter was launched from the repo root or ml-models/notebooks.
HERE = Path.cwd().resolve()
candidates = [
    HERE / 'ml-models',
    HERE.parent,
    HERE.parent.parent,
]
ML_ROOT = next((p for p in candidates if (p / 'data' / 'raw').is_dir()), None)
if ML_ROOT is None:
    raise FileNotFoundError('Could not locate ml-models/data/raw from the current notebook directory.')

RAW_DIR = ML_ROOT / 'data' / 'raw' / 'sharepoint_data' / 'Fruit_Images'
REVIEW_DIR = ML_ROOT / 'data' / 'manifests'
REVIEW_PATH = REVIEW_DIR / 'sharepoint_label_audit_20.csv'

print(f'ML root: {ML_ROOT}')
print(f'Raw image directory: {RAW_DIR}')
print(f'Raw directory exists: {RAW_DIR.is_dir()}')

In [ ]:
# Register HEIC support. From the ml-models project, install with: uv add --dev jupyter pandas matplotlib pillow pillow-heif
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIC_AVAILABLE = True
except ImportError:
    HEIC_AVAILABLE = False
    print('HEIC support is unavailable. Install pillow-heif and rerun this cell.')

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f'Missing raw directory: {RAW_DIR}')

heic_files = sorted(RAW_DIR.glob('*.HEIC'), key=lambda p: p.name.lower())
print(f'Found {len(heic_files):,} HEIC files.')
if len(heic_files) < 20:
    raise ValueError('The audit requires at least 20 HEIC files.')

## Select a spread of files

The selection is deterministic and spread through the sorted collection, so rerunning the notebook audits the same 20 files. Change `SEED` or `SAMPLE_SIZE` if a second audit is needed.

In [ ]:
SAMPLE_SIZE = 20
SEED = 4914
rng = random.Random(SEED)

# Take evenly spaced positions, with a seeded one-position jitter so the sample is not always boundary files.
positions = []
for i in range(SAMPLE_SIZE):
    start = round(i * len(heic_files) / SAMPLE_SIZE)
    end = round((i + 1) * len(heic_files) / SAMPLE_SIZE)
    positions.append(rng.randrange(start, max(start + 1, end)))
sample_files = [heic_files[i] for i in positions]

sample_files

In [ ]:
def open_for_display(path: Path) -> Image.Image:
    # exif_transpose applies the camera orientation before display or later cropping.
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def companion_mov(path: Path):
    companion = path.with_suffix('.MOV')
    return str(companion.relative_to(RAW_DIR)) if companion.exists() else ''

# Optional OCR: pytesseract is intentionally optional because OCR is a draft only.
try:
    import pytesseract
    OCR_BACKEND = 'pytesseract'
except ImportError:
    pytesseract = None
    OCR_BACKEND = 'unavailable'

def ocr_text(image: Image.Image) -> str:
    if pytesseract is None:
        return ''
    try:
        return pytesseract.image_to_string(image).strip()
    except Exception as exc:
        return f'[OCR error: {exc}]'

print(f'OCR backend: {OCR_BACKEND}')
if OCR_BACKEND == 'unavailable':
    print('Optional install: uv add --dev pytesseract (and install the tesseract executable separately).')

In [ ]:
# Render the sample. OCR text is shown below each image when a backend is available.
rows, cols = 4, 5
fig, axes = plt.subplots(rows, cols, figsize=(18, 15))
review_records = []

for ax, path in zip(axes.flat, sample_files):
    try:
        image = open_for_display(path)
        text = ocr_text(image)
        ax.imshow(image)
        ax.set_title(path.name, fontsize=9)
        ax.set_xlabel(text[:140] if text else 'OCR unavailable', fontsize=7)
        review_records.append({
            'sample_id': path.stem,
            'raw_path': str(path.relative_to(ML_ROOT)),
            'companion_mov': companion_mov(path),
            'sha256': sha256(path),
            'width': image.width,
            'height': image.height,
            'ocr_text': text,
            'variety_present': '',
            'variety_raw': '',
            'specimen_present': '',
            'specimen_raw': '',
            'note_readable': '',
            'view_type': '',
            'review_notes': '',
        })
    except Exception as exc:
        ax.set_title(f'{path.name} (ERROR)', fontsize=9)
        ax.text(0.5, 0.5, str(exc), ha='center', va='center', wrap=True)
    ax.axis('off')

for ax in axes.flat[len(sample_files):]:
    ax.axis('off')

fig.suptitle('SharePoint HEIC label audit sample', fontsize=16)
fig.tight_layout()
plt.show()

## Human review

Fill the blank fields below after inspecting the grid. Use `yes`/`no` for the presence/readability fields and use `exterior`, `interior`, or `unknown` for `view_type`. Preserve the exact handwriting in the `*_raw` fields; do not normalize cultivar names here.

In [ ]:
review_df = pd.DataFrame(review_records)
pd.set_option('display.max_colwidth', 80)
review_df[['sample_id', 'ocr_text', 'variety_present', 'variety_raw', 'specimen_present', 'specimen_raw', 'note_readable', 'view_type', 'review_notes']]

In [ ]:
# After editing review_df (for example, with review_df.at[row, 'variety_raw'] = 'Paheri'), export the audit.
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
review_df.to_csv(REVIEW_PATH, index=False)
print(f'Wrote review table to {REVIEW_PATH}')

## What to look for

After reviewing the 20 images, record these conclusions in the notebook output or project notes:

- Are cultivar names present in most images?
- Are specimen codes separate from cultivar names?
- Are the cards readable enough for OCR assistance?
- Do cards move around the frame?
- How many images show intact fruit versus cut fruit?
- Does the sample reveal duplicate views or paired Live Photos?

The review CSV is a pilot annotation artifact. It should not be treated as the full training manifest until labels and crop decisions have been verified across the complete collection.